In [1]:
import pandas as pd
from pathlib import Path

# Measurements

In [2]:
old_data = Path('data')
measurement_folder_old = old_data / 'irradiation_measurements'
mak_measurement_old = measurement_folder_old / 'MAK_physics_dept'
ministry_energy_measurement_old = measurement_folder_old / 'ministry_energy_ug'
CBE_measurement_old = measurement_folder_old / 'CBE_Data'

new_data = Path('Data')
measurement_folder_new = new_data / 'irradiation_measurements'
mak_measurement_new = measurement_folder_new / 'MAK_physics_dept'
ministry_energy_measurement_new = measurement_folder_new / 'ministry_energy_ug'
CBE_measurement_new = measurement_folder_new / 'CBE_Data'

In [ ]:
def read_and_normalize_csvs(filepaths, country='Unidentified', coordinates_dict=None):
    """
    Read and normalize CSV files with solar irradiation data.
    
    Args:
        filepaths: List of file paths or single file path to read
        country: Country identifier for the data
        coordinates_dict: Dictionary mapping location names to (latitude, longitude) tuples
    
    Returns:
        Pandas DataFrame with normalized data including location coordinates
    """
    dataframes = []
    
    if coordinates_dict is None:
        coordinates_dict = {}
    
    for file_path in [filepaths] if not isinstance(filepaths, list) else filepaths:
        df = pd.read_csv(file_path, index_col=False)
        
        # Extract location
        if 'location' in df.columns:
            location = country +  '_' + df['location'][0].lower()
        else:
            location = country
            
        # Identify datetime column
        if 'Day' in df.columns:
            df['datetime'] = pd.to_datetime(df['Day'], errors="coerce")
        elif 'MEASURE_DATE' in df.columns:
            df['datetime'] = pd.to_datetime(df['MEASURE_DATE'], errors='coerce')
        elif 'datetime' in df.columns:
            df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
        else:
            print(f"Warning: No recognized datetime column in {file_path.name}")
            continue

        # Extract GHI column
        if 'GHI_1 (kWh/m2/day)' in df.columns:
            df = df[['datetime', 'GHI_1 (kWh/m2/day)']]
            df.rename(columns={'GHI_1 (kWh/m2/day)': 'ghi'}, inplace=True)
            df['ghi'] = df['ghi'] * 1000
        elif 'MEASURE_VALUE' in df.columns:
            df = df[['datetime', 'MEASURE_VALUE']]
            df.rename(columns={'MEASURE_VALUE': 'ghi'}, inplace=True)
        elif 'GHI (W/m2)' in df.columns:
            df.index = df['datetime']
            df['ghi'] = df['GHI (W/m2)'].resample('D').sum()
            df.dropna(inplace=True)
            df.sort_index(inplace=True)
            df = df[['datetime', 'ghi']]
        else:
            print(f"Warning: No GHI column found in {file_path.name}")
            continue
        
        df['location'] = location
        
        # Add latitude and longitude if available in the coordinates dictionary
        if location in coordinates_dict:
            df['latitude'] = coordinates_dict[location][0]
            df['longitude'] = coordinates_dict[location][1]
        # Check if coordinates are in the original dataframe
        elif 'latitude' in df.columns and 'longitude' in df.columns:
            lat = df['latitude'].iloc[0]
            lon = df['longitude'].iloc[0]
            df['latitude'] = lat
            df['longitude'] = lon
        else:
            df['latitude'] = None
            df['longitude'] = None
            print(f"Warning: No coordinates found for location: {location}")
        
        df.dropna(subset=['datetime', 'ghi'], inplace=True)
        df = df[df['ghi'] > 0]
        dataframes.append(df)
        
    return pd.concat(dataframes, ignore_index=True) if dataframes else None

## Ministry 

In [3]:
soroti_measurements_old = ministry_energy_measurement_old / 'Soroti'
wadelai_measurements_old = ministry_energy_measurement_old / 'Wadelai'

In [4]:
soroti_filepaths = list(soroti_measurements_old.glob('*.csv'))
wadelai_filepaths = list(wadelai_measurements_old.glob('*.csv'))

In [5]:
# coordinates for soroti and wadelai
soroti_coords = (1.7153, 33.6186)
wadelai_coords = (2.7333, 31.4000)

In [6]:
# soroti = read_and_normalize_csvs(soroti_filepaths, 'soroti')
soroti = pd.read_csv(ministry_energy_measurement_new / 'soroti.csv')
soroti['latitude'] = soroti_coords[0]
soroti['longitude'] = soroti_coords[1]
soroti.to_csv(ministry_energy_measurement_new / 'soroti.csv', index=False)

In [17]:
# wadelai = read_and_normalize_csvs(wadelai_filepaths, 'wadelai')
wadelai = pd.read_csv(ministry_energy_measurement_new / 'wadelai.csv')
wadelai['latitude'] = wadelai_coords[0]
wadelai['longitude'] = wadelai_coords[1]
wadelai.to_csv(ministry_energy_measurement_new / 'wadelai.csv', index=False)
# wadelai.to_csv(ministry_energy_measurement_new / 'wadelai.csv', index=False)

## CBE

In [7]:
egypt_measurements = CBE_measurement_old / 'Egypt'
ghana_measurements = CBE_measurement_old / 'Ghana'
kenya_measurements = CBE_measurement_old / 'Kenya'
madagascar_measurements = CBE_measurement_old / 'Madagascar'
nigeria_measurements = CBE_measurement_old / 'Nigeria'
somalia_measurements = CBE_measurement_old / 'Somalia'

In [8]:
egypt_filepaths = list(egypt_measurements.glob('*.csv'))
ghana_filepaths = list(ghana_measurements.glob('*.csv'))
kenya_filepaths = list(kenya_measurements.glob('*.csv'))
madagascar_filepaths = list(madagascar_measurements.glob('*.csv'))
nigeria_filepaths = list(nigeria_measurements.glob('*.csv'))
somalia_filepaths = list(somalia_measurements.glob('*.csv'))

In [9]:
coordinates_dict = {
    'egypt_location1': (29.887561, 32.460447),
    'ghana_location1': (5.645759, -0.105223),
    'ghana_location2': (5.645686, 0.008678),
    'ghana_location3': (-0.235111, 5.632167),
    'kenya_location1': (0.61, 36.83),
    'kenya_location2': (0.61, 36.8),
    'kenya_location3': (-0.469974, 35.181874),
    'kenya_location4': (-1.232761, 36.878509),
    'kenya_location5': (-0.22, 35.86),
    'kenya_location6': (-0.4, 35.92),
    'kenya_location7': (-0.27, 35.85),
    'kenya_location8': (-0.982, 37.0539),
    'kenya_location9': (-0.227185, 35.884984),
    'kenya_location10': (-1.451111, 36.975687),
    'kenya_location11': (-1.494448, 37.057094),
    'kenya_location12': (-0.224, 35.88),
    'kenya_location13': (-1.491302, 37.052862),
    'kenya_location14': (-1.451315, 36.973008),
    'madagascar_location1': (-25.053564, 46.947956),
    'madagascar_location2': (-23.99, 45.12),
    'nigeria_location1': (9.076439, 7.425385),
    'nigeria_location2': (6.448858, 7.392609),
    'nigeria_location3': (7.38, 3.97),
    'somalia_location1': (3.107191, 43.637153)
}

In [ ]:
# egypt = read_and_normalize_csvs(egypt_filepaths, 'egypt', coordinates_dict)
# ghana = read_and_normalize_csvs(ghana_filepaths, 'ghana', coordinates_dict)
# kenya = read_and_normalize_csvs(kenya_filepaths, 'kenya', coordinates_dict)
# madagascar = read_and_normalize_csvs(madagascar_filepaths, 'madagascar', coordinates_dict)
# nigeria = read_and_normalize_csvs(nigeria_filepaths, 'nigeria', coordinates_dict)
# somalia = read_and_normalize_csvs(somalia_filepaths, 'somalia', coordinates_dict)

# egypt.to_csv(CBE_measurement_new / 'egypt.csv', index=False)
# ghana.to_csv(CBE_measurement_new / 'ghana.csv', index=False)
# kenya.to_csv(CBE_measurement_new / 'kenya.csv', index=False)
# nigeria.to_csv(CBE_measurement_new / 'nigeria.csv', index=False)
# somalia.to_csv(CBE_measurement_new / 'somalia.csv', index=False)
# madagascar.to_csv(CBE_measurement_new / 'madagascar.csv', index=False)

egypt = pd.read_csv(CBE_measurement_new / 'egypt.csv')

for key, value in coordinates_dict.items():
    if key in egypt['location'].values:
        egypt.loc[egypt['location'] == key, 'latitude'] = value[0]
        egypt.loc[egypt['location'] == key, 'longitude'] = value[1]
    
egypt.to_csv(CBE_measurement_new / 'egypt.csv', index=False)



In [26]:
ghana = pd.read_csv(CBE_measurement_new / 'ghana.csv')

for key, value in coordinates_dict.items():
    if key in ghana['location'].values:
        ghana.loc[ghana['location'] == key, 'latitude'] = value[0]
        ghana.loc[ghana['location'] == key, 'longitude'] = value[1] 
    
ghana.to_csv(CBE_measurement_new / 'ghana.csv', index=False)

In [29]:
kenya = pd.read_csv(CBE_measurement_new / 'kenya.csv')

for key, value in coordinates_dict.items():
    if key in kenya['location'].values:
        kenya.loc[kenya['location'] == key, 'latitude'] = value[0]
        kenya.loc[kenya['location'] == key, 'longitude'] = value[1]

kenya.to_csv(CBE_measurement_new / 'kenya.csv', index=False)

In [30]:
madagascar = pd.read_csv(CBE_measurement_new / 'madagascar.csv')

for key, value in coordinates_dict.items():
    if key in madagascar['location'].values:
        madagascar.loc[madagascar['location'] == key, 'latitude'] = value[0]
        madagascar.loc[madagascar['location'] == key, 'longitude'] = value[1]

madagascar.to_csv(CBE_measurement_new / 'madagascar.csv', index=False)

In [31]:
nigeria = pd.read_csv(CBE_measurement_new / 'nigeria.csv')

for key, value in coordinates_dict.items():
    if key in nigeria['location'].values:
        nigeria.loc[nigeria['location'] == key, 'latitude'] = value[0]
        nigeria.loc[nigeria['location'] == key, 'longitude'] = value[1]

nigeria.to_csv(CBE_measurement_new / 'nigeria.csv', index=False)

In [32]:
somalia = pd.read_csv(CBE_measurement_new / 'somalia.csv')

for key, value in coordinates_dict.items():
    if key in somalia['location'].values:
        somalia.loc[somalia['location'] == key, 'latitude'] = value[0]
        somalia.loc[somalia['location'] == key, 'longitude'] = value[1]

somalia.to_csv(CBE_measurement_new / 'somalia.csv', index=False)

## Mak

In [124]:
kampala_path = mak_measurement_old / "kampala_data.csv"
lira_path = mak_measurement_old / "lira_data.csv"
mbarara_path = mak_measurement_old / "mbarara_data.csv"
tororo_path = mak_measurement_old / "tororo_data.csv"


In [185]:
kampala = read_and_normalize_csvs(mbarara_path)
kampala.tail()

,datetime,ghi,location
1491,2016-04-19,88.8693,Unidentified
1492,2016-04-20,17692.4132,Unidentified
1493,2016-04-21,35997.7100,Unidentified
1494,2016-04-22,70402.5850,Unidentified
1495,2016-04-23,4472.4340,Unidentified


In [183]:
kampala = read_and_normalize_csvs(kampala_path, 'kampala')
lira = read_and_normalize_csvs(lira_path, 'lira')
tororo = read_and_normalize_csvs(tororo_path, 'tororo')

lira.to_csv(mak_measurement_new / 'lira.csv', index=False)
tororo.to_csv(mak_measurement_new / 'tororo.csv', index=False)
kampala.to_csv(mak_measurement_new / 'kampala.csv', index=False)

# Estimates

In [2]:
old_data = Path('data')
estimates_folder_old = old_data / 'irradiation_estimates'
mak_estimates_old = estimates_folder_old / 'MAK_physics_dept'
ministry_energy_estimates_old = estimates_folder_old / 'ministry_energy_ug'
CBE_estimates_old = estimates_folder_old / 'CBE_Data'

new_data = Path('Data')
estimates_folder_new = new_data / 'irradiation_estimates'
mak_estimates_new = estimates_folder_new / 'MAK_physics_dept'
ministry_energy_estimates_new = estimates_folder_new / 'ministry_energy_ug'
CBE_estimates_new = estimates_folder_new / 'CBE_Data'

In [5]:
def wrangle_solcast_data(path):
    df = pd.read_csv(path)
    columns_to_keep = ['air_temp', 'albedo', 'clearsky_dhi', 'clearsky_dni', 'clearsky_ghi', 'clearsky_gti', 'dhi', 'dni', 'ghi', 'gti', 'period_end', 'period']
    df = df[columns_to_keep]
    df['period_end'] = pd.to_datetime(df['period_end']).dt.tz_convert('Africa/Nairobi')
    df.set_index('period_end', inplace=True)

    df.drop(columns=['period', 'albedo'], inplace=True)

    irr_cols = df.columns[df.columns != 'air_temp']
    df_temp = df[['air_temp']].resample('D').mean()
    df_irr = df[irr_cols].resample('D').sum()
    df = df_temp.join(df_irr)

    df.index = pd.to_datetime(df.index).strftime('%Y-%m-%d')

    df.columns = [
    'air_temp (°C)',
    'clearsky_dhi (Wh/m2/day)',
    'clearsky_dni (Wh/m2/day)',
    'clearsky_ghi (Wh/m2/day)',
    'clearsky_gti (Wh/m2/day)',
    'dhi (Wh/m2/day)',
    'dni (Wh/m2/day)',
    'ghi (Wh/m2/day)',
    'gti (Wh/m2/day)'
    ]

    df.rename(columns={
        'clearsky_dhi (Wh/m2/day)': 'Clearsky DHI (Wh/m2/day)',
        'clearsky_dni (Wh/m2/day)': 'Clearsky DNI (Wh/m2/day)',
        'clearsky_ghi (Wh/m2/day)': 'Clearsky GHI (Wh/m2/day)',
        'clearsky_gti (Wh/m2/day)': 'Clearsky GTI (Wh/m2/day)',
        'dhi (Wh/m2/day)': 'DHI (Wh/m2/day)',
        'dni (Wh/m2/day)': 'DNI (Wh/m2/day)',
        'ghi (Wh/m2/day)': 'GHI (Wh/m2/day)',
        'gti (Wh/m2/day)': 'GTI (Wh/m2/day)'
        }, inplace=True)

    df = df.iloc[:-1] # removing first day of the year
    df.reset_index(inplace=True)
    df.rename(columns={'period_end': 'Day'}, inplace=True)
    return df

In [6]:
def wrangle_camsrad_data(path):
    df = pd.read_csv(path, skiprows=42, delimiter=';')

    irra_cols = [
      'TOA',
      'GHI',
      'DHI',
      'BHI',
      'BNI',
      'Clear sky GHI',
      'Clear sky DHI',
      'Clear sky BHI',
      'Clear sky BNI']

    df.rename(columns={
      'TOA': 'TAO (Wh/m2/day)',
      'GHI': 'GHI (Wh/m2/day)',
      'DHI': 'DHI (Wh/m2/day)',
      'BHI': 'BHI (Wh/m2/day)',
      'BNI': 'DNI (Wh/m2/day)',
      'Clear sky GHI': 'Clear sky GHI (Wh/m2/day)',
      'Clear sky DHI': 'Clear sky DHI (Wh/m2/day)',
      'Clear sky BHI': 'Clear sky BHI (Wh/m2/day)',
      'Clear sky BNI': 'Clear sky DNI (Wh/m2/day)',
    }, inplace=True)

    df[['Day', 'End Day']] = df['# Observation period'].str.split('/', expand=True)
    df['Day'] = pd.to_datetime(df['Day'])
    df.drop(columns= ['# Observation period', 'End Day'], inplace=True, axis=1)

    return df

In [7]:
def wrangle_NREL_Data(filepath):
    df = pd.read_csv(filepath, skiprows=2)

    df['Date'] = pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour', 'Minute']].assign(Day=lambda x: x.Day.astype(str).str.zfill(2)))
    df = df[['Date'] + list(df.columns[:-1])]
    df.drop(columns=['Year', 'Month', 'Day', 'Hour', 'Minute'], inplace=True)

    df.set_index('Date', inplace=True)
    irr_cols = ['Clearsky DHI', 'Clearsky DNI', 'Clearsky GHI', 'DHI', 'DNI', 'GHI']

    df = df[irr_cols].resample('D').sum()
    df.rename(columns={
        'Clearsky DHI': 'Clearsky DHI (Wh/m2/day)',
        'Clearsky DNI': 'Clearsky DNI (Wh/m2/day)',
        'Clearsky GHI': 'Clearsky GHI (Wh/m2/day)',
        'DHI': 'DHI (Wh/m2/day)',
        'DNI': 'DNI (Wh/m2/day)',
        'GHI': 'GHI (Wh/m2/day)',
    }, inplace=True)
    df.reset_index(inplace=True)
    df.rename(columns={'Date': 'Day'}, inplace=True)
    return df

In [10]:
def read_and_update_csvs(filepaths, country='Unidentified'):
    
    dataframes = []
    
    for file_path in [filepaths] if not isinstance(filepaths, list) else filepaths:
        
        estimator = file_path.name.split('_')[1].lower()[:4]
        df = pd.read_csv(file_path, index_col=False)
        
        # Extract location
        if 'location' in df.columns:
            location = country +  '_' + df['location'][0].lower()
        else:
            location = country
        
        # Identify datetime column   
        if 'Unnamed: 0' in df.columns:
            df['datetime'] = pd.to_datetime(df['Unnamed: 0'], errors="coerce")
        else:
            print(f"Warning: No recognized datetime column in {file_path.name}")
            continue
        
        # Extract GHI column
        if 'ghi' in df.columns:
            df.index = df['datetime']
            df['ghi'] = df['ghi'].resample('D').sum().round(3)
            df['datetime'] = df.index
            df['datetime'] = df['datetime'].dt.date
            df.dropna(inplace=True)
            df.sort_index(inplace=True)
            df = df[['datetime', 'ghi']]
        else:
            print(f"Warning: No GHI column found in {file_path.name}")
            continue
    
        df['estimator'] = estimator
        df['location'] = location
        df.dropna(inplace=True)
        df = df[df['ghi'] > 0]
        dataframes.append(df)
        
    return pd.concat(dataframes, ignore_index=True) if dataframes else None

In [18]:
def extract_ghi(df_1, df_2, estimator='Unidentified'):
    # copy dataframs to avoind editing originals
    df1 = df_1.copy()
    df2 = df_2.copy()
    df1['Day'] = pd.to_datetime(df1['Day'])
    df2['Day'] = pd.to_datetime(df2['Day'])
    df1.set_index('Day', inplace=True)
    df2.set_index('Day', inplace=True)

    df = pd.concat([df1, df2])

    # add estimator to ghi column name
    df = df[['GHI (Wh/m2/day)']]
    df.rename(columns={'GHI (Wh/m2/day)': f'GHI_{estimator}'}, inplace=True)
    return df

In [51]:

def read_and_update_csvs(filepaths, country='Unidentified'):
    
    dataframes = []
    
    for file_path in [filepaths] if not isinstance(filepaths, list) else filepaths:
        
        estimator = file_path.name.split('_')[0].lower()
        df = pd.read_csv(file_path, index_col=False)
        
        # Extract location
        if 'location' in df.columns:
            location = country +  '_' + df['location'][0].lower()
        elif 'location' or 'Location' in str(file_path):
            location_part = file_path.name.split('_')[1]
            location = country + '_' + location_part.split('.')[0].lower()
        else:
            location = country
        
        # Identify datetime column   
        if 'Time' in df.columns:
            df['datetime'] = pd.to_datetime(df['Time'], errors="coerce")
        elif 'Date' in df.columns:
            df['datetime'] = pd.to_datetime(df['Date'], errors='coerce')
        else:
            print(f"Warning: No recognized datetime column in {file_path.name}")
            continue
        
        # Extract GHI column
        if 'GHI (Wh/m2/day)' in df.columns:
            df.rename(columns={'GHI (Wh/m2/day)': 'ghi'}, inplace=True)
            df.index = df['datetime']
            df['ghi'] = df['ghi'].resample('D').sum().round(3)
            df['datetime'] = df.index
            df['datetime'] = df['datetime'].dt.date
            df.dropna(inplace=True)
            df.sort_index(inplace=True)
            df = df[['datetime', 'ghi']]
        elif 'GHI (W/m2)' in df.columns:
            df.rename(columns={'GHI (W/m2)': 'ghi'}, inplace=True)
            df.index = df['datetime']
            df['ghi'] = df['ghi'].resample('D').sum().round(3)
            df['datetime'] = df.index
            df['datetime'] = df['datetime'].dt.date
            df.dropna(inplace=True)
            df.sort_index(inplace=True)
            df = df[['datetime', 'ghi']]
        else:
            print(f"Warning: No GHI column found in {file_path.name}")
            continue
    
        df['estimator'] = estimator
        df['location'] = location
        df.dropna(inplace=True)
        df = df[df['ghi'] > 0]
        dataframes.append(df)
        
    return pd.concat(dataframes, ignore_index=True) if dataframes else None

## Ministry

In [3]:
soroti_estimates_old = ministry_energy_estimates_old / 'Soroti'
wadelai_estimates_old = ministry_energy_estimates_old / 'Wadelai'

In [13]:
soroti_filepaths = list(soroti_estimates_old.glob('*.csv'))
wadelai_filepaths = list(wadelai_estimates_old.glob('*csv'))
ministry_files = soroti_filepaths + wadelai_filepaths

ministry_files.pop(0)
for file in ministry_files:
    print(file.name)

Soroti_CAMS-RAD2020.csv
Soroti_CAMS-RAD2021.csv
Soroti_NREL2020.csv
Soroti_NREL2021.csv
Soroti_solcast2020.csv
Soroti_solcast2021.csv
Wadelai_CAMS-RAD2020.csv
Wadelai_CAMS-RAD2021.csv
Wadelai_NREL2020.csv
Wadelai_NREL2021.csv
Wadelai_solcast2020.csv
Wadelai_solcast2021.csv


In [14]:
soroti_cams2020 = wrangle_camsrad_data(ministry_files[0])
soroti_cams2021 = wrangle_camsrad_data(ministry_files[1])
soroti_nrel2020 = wrangle_NREL_Data(ministry_files[2])
soroti_nrel2021 = wrangle_NREL_Data(ministry_files[3])
soroti_solcast2020 = wrangle_solcast_data(ministry_files[4])
soroti_solcast2021 = wrangle_solcast_data(ministry_files[5])

wadelai_cams2020 = wrangle_camsrad_data(ministry_files[6])
wadelai_cams2021 = wrangle_camsrad_data(ministry_files[7])
wadelai_nrel2020 = wrangle_NREL_Data(ministry_files[8])
wadelai_nrel2021 = wrangle_NREL_Data(ministry_files[9])
wadelai_solcast2020 = wrangle_solcast_data(ministry_files[10])
wadelai_solcast2021 = wrangle_solcast_data(ministry_files[11])

In [22]:
soroti_cams = extract_ghi(soroti_cams2020, soroti_cams2021, 'cams')
soroti_nrel = extract_ghi(soroti_nrel2020, soroti_nrel2021, 'nrel')
soroti_solcast = extract_ghi(soroti_solcast2020, soroti_solcast2021, 'solcast')

soroti_cams.to_csv(ministry_energy_estimates_new / 'soroti_cams.csv')
soroti_nrel.to_csv(ministry_energy_estimates_new / 'soroti_nrel.csv')
soroti_solcast.to_csv(ministry_energy_estimates_new / 'soroti_solcast.csv')

In [23]:
wadelai_cams = extract_ghi(wadelai_cams2020, wadelai_cams2021, 'cams')
wadelai_nrel = extract_ghi(wadelai_nrel2020, wadelai_nrel2021, 'nrel')
wadelai_solcast = extract_ghi(wadelai_solcast2020, wadelai_solcast2021, 'solcast')

wadelai_cams.to_csv(ministry_energy_estimates_new / 'wadelai_cams.csv')
wadelai_nrel.to_csv(ministry_energy_estimates_new / 'wadelai_nrel.csv')
wadelai_solcast.to_csv(ministry_energy_estimates_new / 'wadelai_solcast.csv')

## CBE_Estimates

In [24]:
egypt_estimates = CBE_estimates_old / 'Egypt'
ghana_estimates = CBE_estimates_old / 'Ghana'
madagascar_estimates = CBE_estimates_old / 'Madagascar'
nigeria_estimates = CBE_estimates_old / 'Nigeria'
somalia_estimates = CBE_estimates_old / 'Somalia'

In [25]:
egypt_filepaths = list(egypt_estimates.glob('*.csv'))
ghana_filepaths = list(ghana_estimates.glob('*.csv'))
madagascar_filepaths = list(madagascar_estimates.glob('*.csv'))
nigeria_filepaths = list(nigeria_estimates.glob('*.csv'))
somalia_filepaths = list(somalia_estimates.glob('*.csv'))

In [26]:
egypt_filepaths

[PosixPath('data/irradiation_estimates/CBE_Data/Egypt/CAMSRAD_location1.csv'),
 PosixPath('data/irradiation_estimates/CBE_Data/Egypt/Solcast_Location1_29.887561_32.460447_fixed_23_180_PT60M.csv')]

In [53]:
egypt_cams = read_and_update_csvs(egypt_filepaths[0], 'egypt')
egypt_solcast = read_and_update_csvs(egypt_filepaths[1], 'egypt')

egypt_cams.to_csv(CBE_estimates_new / 'egypt_cams.csv', index=False)
egypt_solcast.to_csv(CBE_estimates_new / 'egypt_solcast.csv', index=False)

In [66]:
ghana_filepaths[:3]

[PosixPath('data/irradiation_estimates/CBE_Data/Ghana/CAMSRAD_location1.csv'),
 PosixPath('data/irradiation_estimates/CBE_Data/Ghana/CAMSRAD_location2.csv'),
 PosixPath('data/irradiation_estimates/CBE_Data/Ghana/CAMSRAD_location3.csv')]

In [62]:
ghana_cams = read_and_update_csvs(ghana_filepaths[:3], 'ghana')
ghana_solcast = read_and_update_csvs(ghana_filepaths[3:], 'ghana')

ghana_cams.to_csv(CBE_estimates_new / 'ghana_cams.csv', index=False)
ghana_solcast.to_csv(CBE_estimates_new / 'ghana_solcast.csv', index=False)

In [67]:
madagascar_filepaths[:2]

[PosixPath('data/irradiation_estimates/CBE_Data/Madagascar/CAMSRAD_location1.csv'),
 PosixPath('data/irradiation_estimates/CBE_Data/Madagascar/CAMSRAD_location2.csv')]

In [69]:
madagascar_cams = read_and_update_csvs(madagascar_filepaths[:2], 'madagascar')
madagascar_solcast = read_and_update_csvs(madagascar_filepaths[2:], 'madagascar')

madagascar_cams.to_csv(CBE_estimates_new / 'madagascar_cams.csv', index=False)
madagascar_solcast.to_csv(CBE_estimates_new / 'madagascar_solcast.csv', index=False)

In [73]:
nigeria_cams = read_and_update_csvs(nigeria_filepaths, 'nigeria')

nigeria_cams.to_csv(CBE_estimates_new / 'nigeria_cams.csv', index=False)

In [76]:
somalia_cams = read_and_update_csvs(somalia_filepaths[0], 'somalia')
somalia_solcast = read_and_update_csvs(somalia_filepaths[1], 'somalia') 

somalia_cams.to_csv(CBE_estimates_new / 'somalia_cams.csv', index=False)
somalia_solcast.to_csv(CBE_estimates_new / 'somalia_solcast.csv', index=False)